In [2]:
# CELL 1: Fixed Installation for Google Colab

import os
import sys

print("🔧 Installing liboqs with shared library support...")
print("This takes 3-4 minutes...\n")

# Install system dependencies
print("Step 1/5: Installing packages...")
!apt-get update -qq
!apt-get install -y cmake ninja-build git build-essential

# Clone liboqs
print("\nStep 2/5: Downloading liboqs...")
!rm -rf liboqs
!git clone --depth=1 https://github.com/open-quantum-safe/liboqs.git

# Build liboqs WITH SHARED LIBRARY (this is the fix!)
print("\nStep 3/5: Building liboqs with shared library...")
!cd liboqs && mkdir -p build && cd build && cmake -GNinja -DCMAKE_INSTALL_PREFIX=/usr/local -DBUILD_SHARED_LIBS=ON .. && ninja

# Install liboqs
print("\nStep 4/5: Installing liboqs...")
!cd liboqs/build && ninja install
!ldconfig

# Install Python wrapper
print("\nStep 5/5: Installing Python packages...")
!pip install -q liboqs-python cryptography matplotlib pandas

# Set library path
os.environ['LD_LIBRARY_PATH'] = '/usr/local/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

print("\n" + "="*70)
print("✅ INSTALLATION COMPLETE!")
print("="*70)

# Verify installation
try:
    import oqs
    print(f"\n✓ liboqs version: {oqs.oqs_version()}")
    print(f"✓ liboqs-python version: {oqs.oqs_python_version()}")
    print(f"✓ Python version: {sys.version.split()[0]}")
    print("\n🎉 Ready to start coding!")
    print("="*70)
except Exception as e:
    print(f"\n❌ Error: {e}")

🔧 Installing liboqs with shared library support...
This takes 3-4 minutes...

Step 1/5: Installing packages...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
Suggested packages:
  gettext-base git-daemon-run | git-daemon-sysvinit git-doc git-email git-gui
  gitk gitweb git-cvs git-mediawiki git-svn
The following NEW packages will be installed:
  ninja-build
The following packages will be upgraded:
  git
1 upgraded, 1 newly installed, 0 to remove and 103 not upgraded.
Need to get 3,285 kB of archives.
After this operation, 358 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64

/usr/local/lib/python3.12/dist-packages/oqs/__init__.py:1: UserWarning: liboqs version (major, minor) 0.15.0 differs from liboqs-python version 0.14.1
  from oqs.oqs import (


In [3]:
# CELL 2: List Available Post-Quantum Algorithms

import oqs

print("\n" + "="*70)
print("POST-QUANTUM KEY ENCAPSULATION MECHANISMS (KEMs)")
print("="*70)

kems = oqs.get_enabled_kem_mechanisms()
print(f"\nTotal KEMs available: {len(kems)}\n")

nist_kems = [k for k in kems if 'Kyber' in k or 'ML-KEM' in k]
print("NIST Standardized KEMs:")
for kem in nist_kems:
    print(f"  ✓ {kem}")

print("\n" + "="*70)
print("POST-QUANTUM SIGNATURE SCHEMES")
print("="*70)

sigs = oqs.get_enabled_sig_mechanisms()
print(f"\nTotal Signature schemes available: {len(sigs)}\n")

nist_sigs = [s for s in sigs if 'Dilithium' in s or 'ML-DSA' in s or 'Falcon' in s]
print("NIST Standardized Signatures:")
for sig in nist_sigs[:10]:
    print(f"  ✓ {sig}")

print("\n" + "="*70)



POST-QUANTUM KEY ENCAPSULATION MECHANISMS (KEMs)

Total KEMs available: 38

NIST Standardized KEMs:
  ✓ Kyber512
  ✓ Kyber768
  ✓ Kyber1024
  ✓ ML-KEM-512
  ✓ ML-KEM-768
  ✓ ML-KEM-1024

POST-QUANTUM SIGNATURE SCHEMES

Total Signature schemes available: 209

NIST Standardized Signatures:
  ✓ ML-DSA-44
  ✓ ML-DSA-65
  ✓ ML-DSA-87
  ✓ Falcon-512
  ✓ Falcon-1024
  ✓ Falcon-padded-512
  ✓ Falcon-padded-1024



In [4]:
# CELL 3: Post-Quantum Key Exchange Demo

import oqs

print("\n" + "="*70)
print("DEMO: POST-QUANTUM KEY EXCHANGE (ML-KEM-768)")
print("="*70 + "\n")

print("👩 ALICE: Generating key pair...")
with oqs.KeyEncapsulation("ML-KEM-768") as alice:
    alice_public_key = alice.generate_keypair()
    print(f"  ✓ Public key size: {len(alice_public_key)} bytes")
    print(f"  ✓ Secret key size: {alice.details['length_secret_key']} bytes")

    print("\n👨 BOB: Creating shared secret using Alice's public key...")
    with oqs.KeyEncapsulation("ML-KEM-768") as bob:
        ciphertext, bob_shared_secret = bob.encap_secret(alice_public_key)
        print(f"  ✓ Ciphertext size: {len(ciphertext)} bytes")
        print(f"  ✓ Bob's shared secret: {bob_shared_secret[:16].hex()}...")

    print("\n👩 ALICE: Recovering shared secret from ciphertext...")
    alice_shared_secret = alice.decap_secret(ciphertext)
    print(f"  ✓ Alice's shared secret: {alice_shared_secret[:16].hex()}...")

    print("\n" + "="*70)
    print("VERIFICATION")
    print("="*70)

    if alice_shared_secret == bob_shared_secret:
        print("✅ SUCCESS!")
        print("Both parties share identical quantum-safe key!")
    else:
        print("❌ FAILED!")

print("\n" + "="*70)



DEMO: POST-QUANTUM KEY EXCHANGE (ML-KEM-768)

👩 ALICE: Generating key pair...
  ✓ Public key size: 1184 bytes
  ✓ Secret key size: 2400 bytes

👨 BOB: Creating shared secret using Alice's public key...
  ✓ Ciphertext size: 1088 bytes
  ✓ Bob's shared secret: 539b71f1e4d2c5d372da73182778fde3...

👩 ALICE: Recovering shared secret from ciphertext...
  ✓ Alice's shared secret: 539b71f1e4d2c5d372da73182778fde3...

VERIFICATION
✅ SUCCESS!
Both parties share identical quantum-safe key!



In [5]:
# CELL 4: Post-Quantum Digital Signatures Demo

import oqs

print("\n" + "="*70)
print("DEMO: POST-QUANTUM DIGITAL SIGNATURES (ML-DSA-65)")
print("="*70 + "\n")

message = b"This is a critical software update package v2.1.0"
print(f"📄 Message: '{message.decode()}'\n")

print("🔑 Generating signing keypair...")
with oqs.Signature("ML-DSA-65") as signer:
    public_key = signer.generate_keypair()
    print(f"  ✓ Public key: {len(public_key)} bytes")

    print("\n✍️ Signing the message...")
    signature = signer.sign(message)
    print(f"  ✓ Signature: {len(signature)} bytes")

    print("\n✅ Verifying with CORRECT message...")
    is_valid = signer.verify(message, signature, public_key)
    print(f"  Result: {'✅ VALID' if is_valid else '❌ INVALID'}")

    print("\n🔍 Verifying with TAMPERED message...")
    tampered = b"This is a critical software update package v2.1.1"
    print(f"  (Changed v2.1.0 → v2.1.1)")

    is_valid_tampered = signer.verify(tampered, signature, public_key)
    print(f"  Result: {'✅ VALID' if is_valid_tampered else '❌ INVALID (Good!)'}")

    if not is_valid_tampered:
        print("\n✅ Tampering detected successfully!")

print("\n" + "="*70)



DEMO: POST-QUANTUM DIGITAL SIGNATURES (ML-DSA-65)

📄 Message: 'This is a critical software update package v2.1.0'

🔑 Generating signing keypair...
  ✓ Public key: 1952 bytes

✍️ Signing the message...
  ✓ Signature: 3309 bytes

✅ Verifying with CORRECT message...
  Result: ✅ VALID

🔍 Verifying with TAMPERED message...
  (Changed v2.1.0 → v2.1.1)
  Result: ❌ INVALID (Good!)

✅ Tampering detected successfully!



In [6]:
# CELL 5: Complete Secure File Transfer System

import oqs
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
import os
import pathlib

class SecureFileTransfer:
    def __init__(self):
        self.kem_alg = "ML-KEM-768"
        self.sig_alg = "ML-DSA-65"
        print("🔐 Secure Transfer System Initialized\n")

    def derive_key(self, secret, salt):
        kdf = HKDF(
            algorithm=hashes.SHA256(),
            length=32,
            salt=salt,
            info=b'pqc-secure-transfer'
        )
        return kdf.derive(secret)

    def sender_prepare(self, filename, receiver_pk):
        print("="*70)
        print("SENDER: Preparing Secure Transfer")
        print("="*70 + "\n")

        with open(filename, 'rb') as f:
            data = f.read()
        print(f"📂 Reading file: {len(data)} bytes\n")

        with oqs.Signature(self.sig_alg) as signer:
            sender_pk = signer.generate_keypair()
            print("🔑 Generated signature keys\n")

            with oqs.KeyEncapsulation(self.kem_alg) as kem:
                kem_ct, secret = kem.encap_secret(receiver_pk)
                print("🤝 Established shared secret\n")

                salt = os.urandom(16)
                key = self.derive_key(secret, salt)

                aesgcm = AESGCM(key)
                nonce = os.urandom(12)
                ciphertext = aesgcm.encrypt(nonce, data, None)
                print(f"🔒 Encrypted: {len(ciphertext)} bytes\n")

                sig = signer.sign(ciphertext)
                print(f"✍️ Signed: {len(sig)} bytes\n")

                return {
                    'kem_ct': kem_ct.hex(),
                    'nonce': nonce.hex(),
                    'ciphertext': ciphertext.hex(),
                    'signature': sig.hex(),
                    'sender_pk': sender_pk.hex(),
                    'filename': pathlib.Path(filename).name,
                    'salt': salt.hex()
                }

    def receiver_process(self, package, receiver_kem):
        print("="*70)
        print("RECEIVER: Processing Package")
        print("="*70 + "\n")

        kem_ct = bytes.fromhex(package['kem_ct'])
        nonce = bytes.fromhex(package['nonce'])
        ciphertext = bytes.fromhex(package['ciphertext'])
        sig = bytes.fromhex(package['signature'])
        sender_pk = bytes.fromhex(package['sender_pk'])
        salt = bytes.fromhex(package['salt'])

        print("🔓 Recovering shared secret\n")
        secret = receiver_kem.decap_secret(kem_ct)

        key = self.derive_key(secret, salt)

        print("✅ Verifying signature...")
        with oqs.Signature(self.sig_alg) as verifier:
            if verifier.verify(ciphertext, sig, sender_pk):
                print("  ✅ VALID - Sender authenticated!\n")
            else:
                print("  ❌ INVALID!\n")
                return None

        print("🔓 Decrypting file...")
        aesgcm = AESGCM(key)
        data = aesgcm.decrypt(nonce, ciphertext, None)
        print(f"  ✓ Decrypted: {len(data)} bytes\n")

        out = f"decrypted_{package['filename']}"
        with open(out, 'wb') as f:
            f.write(data)
        print(f"💾 Saved as: {out}\n")

        return data


# DEMONSTRATION
print("="*70)
print("SECURE FILE TRANSFER DEMONSTRATION")
print("="*70 + "\n")

# Create test file
with open("secret.txt", 'w') as f:
    f.write("TOP SECRET: Quantum-Safe Protocol\n")
    f.write("="*50 + "\n")
    f.write("This file is encrypted with PQC!\n")
    f.write("Protected by ML-KEM and ML-DSA.\n")

print("✓ Created test file\n")

system = SecureFileTransfer()

with oqs.KeyEncapsulation("ML-KEM-768") as receiver_kem:
    receiver_pk = receiver_kem.generate_keypair()

    package = system.sender_prepare("secret.txt", receiver_pk)

    print("="*70)
    print("📡 TRANSMITTING...")
    print("="*70 + "\n")

    decrypted = system.receiver_process(package, receiver_kem)

    if decrypted:
        with open("secret.txt", 'rb') as f:
            original = f.read()

        print("="*70)
        print("FINAL VERIFICATION")
        print("="*70)

        if original == decrypted:
            print("✅ SUCCESS!")
            print("Files match perfectly!")
            print("Transfer completed securely!")

        print("="*70 + "\n")

        print("📄 File contents:")
        print(decrypted.decode())


SECURE FILE TRANSFER DEMONSTRATION

✓ Created test file

🔐 Secure Transfer System Initialized

SENDER: Preparing Secure Transfer

📂 Reading file: 150 bytes

🔑 Generated signature keys

🤝 Established shared secret

🔒 Encrypted: 166 bytes

✍️ Signed: 3309 bytes

📡 TRANSMITTING...

RECEIVER: Processing Package

🔓 Recovering shared secret

✅ Verifying signature...
  ✅ VALID - Sender authenticated!

🔓 Decrypting file...
  ✓ Decrypted: 150 bytes

💾 Saved as: decrypted_secret.txt

FINAL VERIFICATION
✅ SUCCESS!
Files match perfectly!
Transfer completed securely!

📄 File contents:
TOP SECRET: Quantum-Safe Protocol
This file is encrypted with PQC!
Protected by ML-KEM and ML-DSA.



In [7]:
# CELL 6: Performance Comparison (Improved)

import oqs
import time
import pandas as pd
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes

print("="*70)
print("PERFORMANCE: PQC vs CLASSICAL")
print("="*70 + "\n")

iterations = 50
test_message = b"Performance test message"

# ======================
# PQC: ML-KEM-768
# ======================

print("Testing ML-KEM-768 (PQC)...")

with oqs.KeyEncapsulation("ML-KEM-768") as kem:

    start = time.perf_counter()

    for _ in range(iterations):
        pk = kem.generate_keypair()
        ct, sec1 = kem.encap_secret(pk)
        sec2 = kem.decap_secret(ct)

    pqc_time = (time.perf_counter() - start) / iterations
    pqc_key_size = len(pk)
    pqc_ct_size = len(ct)

print(f"  ✓ Avg Time: {pqc_time*1000:.2f} ms\n")


# ======================
# Classical: RSA-2048
# ======================

print("Testing RSA-2048 (Classical)...")

start = time.perf_counter()

for _ in range(iterations):
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048
    )

    public_key = private_key.public_key()

    ciphertext = public_key.encrypt(
        test_message,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

    decrypted = private_key.decrypt(
        ciphertext,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

rsa_time = (time.perf_counter() - start) / iterations
rsa_key_size = 256  # RSA-2048 public modulus ~256 bytes

print(f"  ✓ Avg Time: {rsa_time*1000:.2f} ms\n")


# ======================
# Results Table
# ======================

results = [
    ["ML-KEM-768 (PQC)", f"{pqc_time*1000:.2f}", pqc_key_size, pqc_ct_size, "✅ Yes"],
    ["RSA-2048", f"{rsa_time*1000:.2f}", rsa_key_size, len(ciphertext), "❌ No"]
]

df = pd.DataFrame(
    results,
    columns=["Algorithm", "Time (ms)", "Public Key Size (bytes)", "Ciphertext Size (bytes)", "Quantum-Safe"]
)

print("="*70)
print(df.to_string(index=False))
print("="*70 + "\n")

speed_ratio = rsa_time / pqc_time

if speed_ratio > 1:
    print(f"📊 ML-KEM-768 is {speed_ratio:.2f}x faster than RSA-2048")
else:
    print(f"📊 RSA-2048 is {(1/speed_ratio):.2f}x faster than ML-KEM-768")

print("="*70)


PERFORMANCE: PQC vs CLASSICAL

Testing ML-KEM-768 (PQC)...
  ✓ Avg Time: 0.12 ms

Testing RSA-2048 (Classical)...
  ✓ Avg Time: 128.43 ms

       Algorithm Time (ms)  Public Key Size (bytes)  Ciphertext Size (bytes) Quantum-Safe
ML-KEM-768 (PQC)      0.12                     1184                     1088        ✅ Yes
        RSA-2048    128.43                      256                      256         ❌ No

📊 ML-KEM-768 is 1108.66x faster than RSA-2048


In [8]:
%%writefile secure_transfer.py

import oqs
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import os

class SecureFileTransfer:
    def __init__(self, kem_algorithm="ML-KEM-768"):
        self.kem_algorithm = kem_algorithm

    # Generate keypair for receiver
    def generate_keys(self):
        self.kem = oqs.KeyEncapsulation(self.kem_algorithm)
        public_key = self.kem.generate_keypair()
        return public_key

    # Sender encrypts file
    def encrypt_file(self, file_path, receiver_public_key):
        with oqs.KeyEncapsulation(self.kem_algorithm) as sender_kem:
            ciphertext, shared_secret = sender_kem.encap_secret(receiver_public_key)

        # Read file
        with open(file_path, "rb") as f:
            data = f.read()

        # AES encryption using shared secret
        aesgcm = AESGCM(shared_secret[:32])
        nonce = os.urandom(12)
        encrypted_data = aesgcm.encrypt(nonce, data, None)

        return ciphertext, nonce, encrypted_data

    # Receiver decrypts file
    def decrypt_file(self, ciphertext, nonce, encrypted_data):
        shared_secret = self.kem.decap_secret(ciphertext)

        aesgcm = AESGCM(shared_secret[:32])
        decrypted_data = aesgcm.decrypt(nonce, encrypted_data, None)

        return decrypted_data

Writing secure_transfer.py


In [9]:
%%writefile send.py
from secure_transfer import SecureFileTransfer

print("=== SENDER SIDE ===")

sft = SecureFileTransfer()

# Receiver generates keypair
receiver_public_key = sft.generate_keys()

# Create test file
with open("secret.txt", "w") as f:
    f.write("Quantum Safe File Transfer Test")

ciphertext, nonce, encrypted_data = sft.encrypt_file("secret.txt", receiver_public_key)

print("File Encrypted Successfully ✅")

# Save encrypted package
import pickle
with open("package.pkl", "wb") as f:
    pickle.dump((ciphertext, nonce, encrypted_data), f)

print("Encrypted package saved as package.pkl")


Writing send.py


In [10]:
%%writefile receive.py
from secure_transfer import SecureFileTransfer
import pickle

print("=== RECEIVER SIDE ===")

sft = SecureFileTransfer()

# Generate same keypair (simulate same receiver)
receiver_public_key = sft.generate_keys()

# Load encrypted package
with open("package.pkl", "rb") as f:
    ciphertext, nonce, encrypted_data = pickle.load(f)

decrypted_data = sft.decrypt_file(ciphertext, nonce, encrypted_data)

print("File Decrypted Successfully ✅")
print("Decrypted Content:")
print(decrypted_data.decode())



Writing receive.py


In [11]:
!ls


decrypted_secret.txt  receive.py   secret.txt	       send.py
liboqs		      sample_data  secure_transfer.py


In [12]:
!python send.py
!python receive.py

/usr/local/lib/python3.12/dist-packages/oqs/__init__.py:1: UserWarning: liboqs version (major, minor) 0.15.0 differs from liboqs-python version 0.14.1
  from oqs.oqs import (
=== SENDER SIDE ===
File Encrypted Successfully ✅
Encrypted package saved as package.pkl
/usr/local/lib/python3.12/dist-packages/oqs/__init__.py:1: UserWarning: liboqs version (major, minor) 0.15.0 differs from liboqs-python version 0.14.1
  from oqs.oqs import (
=== RECEIVER SIDE ===
Traceback (most recent call last):
  File "/content/receive.py", line 15, in <module>
    decrypted_data = sft.decrypt_file(ciphertext, nonce, encrypted_data)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/secure_transfer.py", line 37, in decrypt_file
    decrypted_data = aesgcm.decrypt(nonce, encrypted_data, None)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
cryptography.exceptions.InvalidTag


In [13]:
with open("secret.txt","w") as f:
    f.write("Quantum Safe File Transfer Test")